# DARA Phase 3 — Verify Gemini, Gemini CLI, BigQuery, Vertex AI, and optional OpenAI
**Date:** 2026-10-14 | **Google Cloud project:** `third-serenity-428705-g2`.

This notebook is a **non-destructive, low-cost connection test**: it does not create cloud resources, activate APIs or billing, generate AI model responses, or modify GitHub. It writes a uniquely named verification report to the existing Google Drive `MyDrive/DARA_ResearchOS/` folder.

**Separation of access:** ChatGPT ↔ Google Drive/GitHub uses ChatGPT's independently connected apps; Colab ↔ Drive/GCP uses Google authentication; Gemini CLI lives in a *separate Cloud Shell session*. Colab cannot programmatically prove that Gemini's sidebar assistant worked or that Cloud Shell has been authenticated. These checks require manual confirmation. **A test of API metadata is not proof that model generation or billable inference is available.**

Recommended: run each cell in order. Do not paste passwords, OAuth tokens, API keys or private research data into notebook source or outputs.


## 1 — Configure safe verification

The only network API operations are read-only metadata requests. `RUN_API_METADATA_TESTS` can be switched to `False` if you want package/configuration checks only. No model-generation calls exist in this notebook.


In [ ]:
import os, sys, json, subprocess, uuid, platform
from pathlib import Path
from datetime import datetime, timezone
from urllib.request import Request as URLRequest, urlopen

PROJECT_ID = "third-serenity-428705-g2"
GITHUB_REPO = "Saehon/Saeid-Homayoun"
FOLDER = Path("/content/drive/MyDrive/DARA_ResearchOS")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:6]

RUN_API_METADATA_TESTS = True  # read-only metadata checks, no model inference
RUN_OPENAI_MODELS_LIST = True  # only if OPENAI_API_KEY exists in private Colab Secrets

checks = {
    "Colab runtime": {"status": "PASS", "details": "Python executed in Colab" if "google.colab" in sys.modules else "Python executed; Colab environment not yet proven"},
    "Drive readback": {"status": "NOT TESTED"},
    "Cloud credentials": {"status": "NOT TESTED"},
    "GitHub read-only": {"status": "NOT TESTED"},
    "Gemini in Colab UI": {"status": "NOT TESTED — user confirmation needed"},
    "Gemini CLI installation": {"status": "NOT TESTED — separate Cloud Shell session"},
    "Gemini CLI authentication": {"status": "NOT TESTED — separate Cloud Shell session"},
    "BigQuery enabled": {"status": "NOT TESTED"},
    "BigQuery metadata access": {"status": "NOT TESTED"},
    "Vertex AI enabled": {"status": "NOT TESTED"},
    "Vertex AI metadata access": {"status": "NOT TESTED"},
    "Vertex Gemini model inference": {"status": "NOT TESTED — no model call"},
    "OpenAI secret": {"status": "NOT TESTED"},
    "OpenAI metadata API": {"status": "NOT TESTED — no inference"}
}
def record(name, status, details=""):
    checks[name] = {"status": status, "details": details}
    print(f"{name}: {status}" + (f" — {details}" if details else ""))

print("Run:", RUN_ID)
print("Python:", platform.python_version())
print("Target Google Cloud project:", PROJECT_ID)


## 2 — Mount Google Drive and verify readback

You must approve the mount prompt. Only a **new** non-sensitive test text file is added; nothing is overwritten or deleted.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
if not FOLDER.parent.exists():
    raise RuntimeError("Google Drive not available after mount.")
FOLDER.mkdir(parents=True, exist_ok=True)
test_path = FOLDER / f"DARA_PHASE3_DRIVE_TEST_{RUN_ID}.txt"
expected = "DARA Phase 3 Google Drive readback verified: " + RUN_ID
with test_path.open("x", encoding="utf-8") as f:
    f.write(expected)
if test_path.read_text(encoding="utf-8") != expected:
    raise RuntimeError("Drive readback failed")
record("Drive readback", "PASS", test_path.name)


## 3 — Authenticate Google Cloud and select the project

The login prompt is controlled by Google. This step does **not** enable APIs or grant IAM roles.


In [ ]:
from google.colab import auth
import google.auth
auth.authenticate_user()
result = subprocess.run(["gcloud", "config", "set", "project", PROJECT_ID],
                        capture_output=True, text=True)
if result.returncode:
    record("Cloud credentials", "FAIL", "Could not set project: " + result.stderr[-200:])
    raise RuntimeError("gcloud project selection failed")
configured = subprocess.check_output(
    ["gcloud", "config", "get-value", "project"], text=True).strip()
credentials, adc_project = google.auth.default(
    scopes=["https://www.googleapis.com/auth/cloud-platform"])
if configured != PROJECT_ID or credentials is None:
    raise RuntimeError("Cloud credential/project mismatch")
record("Cloud credentials", "PASS", "ADC available; gcloud project " + configured)
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID


## 4 — Verify GitHub as a read-only code bridge

GitHub is used for notebooks and code only; Google Drive receives test reports. This step does not log into GitHub, clone a private repository, commit or push.


In [ ]:
api_url = "https://api.github.com/repos/" + GITHUB_REPO + "/contents/DARA/ARCHITECTURE.md"
try:
    with urlopen(URLRequest(api_url, headers={"User-Agent": "DARA-Phase3-ReadCheck"}), timeout=15) as response:
        metadata = json.load(response)
    if metadata.get("path") != "DARA/ARCHITECTURE.md":
        raise RuntimeError("Unexpected repository path")
    record("GitHub read-only", "PASS", "DARA architecture is readable on GitHub main")
except Exception as error:
    record("GitHub read-only", "NOT VERIFIED", type(error).__name__ + ": " + str(error)[:150])


## 5 — Install read-only service client libraries (Colab runtime only)

These packages do not install on Windows. Installing packages and creating clients do **not** verify model API access.


In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "google-cloud-bigquery", "openai"],
    check=True, timeout=180)
print("BigQuery and OpenAI Python clients available in the Colab session.")


## 6 — Inspect enabled Google APIs without changing them

A successful service list confirms which Google APIs are enabled. If service-list permission is missing, status is **UNKNOWN**, not **DISABLED**. No APIs are enabled by this notebook.


In [ ]:
enabled_services = None
proc = subprocess.run(
    ["gcloud", "services", "list", "--enabled", "--project=" + PROJECT_ID, "--format=json"],
    capture_output=True, text=True, timeout=35
)
if proc.returncode == 0:
    enabled_services = {
        item.get("config", {}).get("name") or item.get("name", "").split("/")[-1]
        for item in json.loads(proc.stdout)
    }
    for label, service in [
        ("BigQuery enabled", "bigquery.googleapis.com"),
        ("Vertex AI enabled", "aiplatform.googleapis.com")
    ]:
        record(label, "ENABLED" if service in enabled_services else "DISABLED",
               "Based on gcloud services list (read-only)")
else:
    record("BigQuery enabled", "UNKNOWN", "Service Usage listing unavailable or unauthorized")
    record("Vertex AI enabled", "UNKNOWN", "Service Usage listing unavailable or unauthorized")
    print("Service listing unavailable; this does NOT establish either API is disabled.")


## 7 — Read-only BigQuery metadata probe

This checks whether the authenticated account can list *visible dataset metadata*. No SQL query, dataset creation or data transfer occurs. An empty list does not imply there are no datasets you could be granted access to.


In [ ]:
if not RUN_API_METADATA_TESTS:
    record("BigQuery metadata access", "SKIPPED", "RUN_API_METADATA_TESTS=False")
elif enabled_services is not None and "bigquery.googleapis.com" not in enabled_services:
    record("BigQuery metadata access", "SKIPPED", "API reported disabled")
else:
    try:
        from google.cloud import bigquery
        bq = bigquery.Client(project=PROJECT_ID, credentials=credentials)
        sample = list(bq.list_datasets(max_results=1, timeout=20))
        record("BigQuery metadata access", "PASS",
               f"Read-only datasets.list succeeded; {len(sample)} visible in limited sample")
    except Exception as error:
        record("BigQuery metadata access", "FAIL OR BLOCKED",
               type(error).__name__ + ": " + str(error)[:180])


## 8 — Read-only Vertex AI metadata probe

This checks the Vertex AI `models.list` metadata endpoint, **not** the Gemini publisher model inference endpoint. A PASS cannot prove Gemini generation is available; there are no paid model calls in this notebook.

Uses `europe-west4` as a probe location; choose another supported project location later if needed.


In [ ]:
if not RUN_API_METADATA_TESTS:
    record("Vertex AI metadata access", "SKIPPED", "RUN_API_METADATA_TESTS=False")
elif enabled_services is not None and "aiplatform.googleapis.com" not in enabled_services:
    record("Vertex AI metadata access", "SKIPPED", "API reported disabled")
else:
    try:
        from google.auth.transport.requests import AuthorizedSession
        session = AuthorizedSession(credentials)
        location = "europe-west4"
        url = (
            f"https://{location}-aiplatform.googleapis.com/v1/"
            f"projects/{PROJECT_ID}/locations/{location}/models?pageSize=1"
        )
        res = session.get(url, timeout=20)
        if res.ok:
            record("Vertex AI metadata access", "PASS",
                   "Vertex models.list metadata responded; Gemini inference NOT tested")
        else:
            record("Vertex AI metadata access", "FAIL OR BLOCKED",
                   "HTTP " + str(res.status_code) + "; check IAM/API/billing and region")
    except Exception as error:
        record("Vertex AI metadata access", "FAIL OR BLOCKED",
               type(error).__name__ + ": " + str(error)[:180])


## 9 — Optional OpenAI GPT API verification from Colab

You **do not need an OpenAI API key** for ChatGPT itself to use its connected Google Drive/GitHub apps.

Only if you want **Colab Python code to invoke OpenAI API**, open **Colab → Secrets** and create `OPENAI_API_KEY`; enable notebook access to it. The cell checks key presence and optionally calls the **read-only `models.list`** endpoint (no text generation). This does not test whether paid inference credits are available. **Never print or save your key**.


In [ ]:
from google.colab import userdata
from openai import OpenAI
secret = None
try:
    secret = userdata.get("OPENAI_API_KEY")
except Exception:
    pass

if not secret:
    record("OpenAI secret", "NOT CONFIGURED", "Optional; add OPENAI_API_KEY in Colab Secrets")
    record("OpenAI metadata API", "NOT TESTED", "No API key present")
else:
    record("OpenAI secret", "PRESENT", "Stored in Colab Secrets; not printed or saved")
    if RUN_OPENAI_MODELS_LIST:
        try:
            client = OpenAI(api_key=secret, timeout=20, max_retries=0)
            response = client.models.list()  # Metadata only; NO inference
            record("OpenAI metadata API", "PASS",
                   f"Authenticated models.list; {len(response.data)} model IDs visible")
            client.close()
        except Exception as error:
            record("OpenAI metadata API", "FAIL OR BLOCKED",
                   type(error).__name__ + "; review Platform account, key and permissions")
    else:
        record("OpenAI metadata API", "SKIPPED", "No network API request selected")
secret = None


## 10 — Record Gemini manual test results without blocking execution

**The notebook now completes this step automatically without asking for any interactive input.** The following status variables default to `None` (NOT VERIFIED). Only change them after actually completing a manual test.

- `GEMINI_COLAB_UI_RESPONDED = True` only after opening the Colab Gemini sidebar, asking a harmless question and receiving a response.
- `GEMINI_CLI_VERSION = "actual version"` only after opening a separate **Google Cloud Shell** tab and running `gemini --version`.
- `GEMINI_CLI_ANSWERED = True` only after voluntarily authenticating Gemini CLI, running `gemini` and receiving an actual response. This interactive CLI request may use quota or incur charges depending on how you sign in.

If you haven't tested them, **leave all three values set to `None`**. The final report will correctly say NOT VERIFIED rather than inventing PASS.

Run this read-only Cloud Shell check separately, if desired:

```bash
gcloud config set project third-serenity-428705-g2
gcloud config get-value project
gemini --version
```

**Google Colab cannot inspect the separate Cloud Shell session or determine whether the Gemini sidebar works.** This step records optional user-confirmed results only; it is not an independent API test.

Proceed immediately to **Step 11 — Save the final evidence report** after this non-blocking cell.


In [ ]:
# NON-BLOCKING MANUAL CONFIRMATIONS
# Leave these default values unless you have independently tested each interface.
# Change 'None' to True or False for an actual completed Gemini Colab/UI or
# Gemini CLI interactive test. Enter the non-secret version string only after
# running 'gemini --version' in a separate Google Cloud Shell terminal.
GEMINI_COLAB_UI_RESPONDED = None        # True / False / None (not tested)
GEMINI_CLI_VERSION = None              # e.g. "0.XX.X" / None
GEMINI_CLI_ANSWERED = None             # True / False / None (not tested)

def manual_status(result, name, what):
    if result is True:
        record(name, "PASS — USER REPORTED", what + "; not independently verified")
    elif result is False:
        record(name, "FAIL — USER REPORTED", what + " test unsuccessful")
    else:
        record(name, "NOT VERIFIED", what + " not tested; no interactive prompt")

manual_status(GEMINI_COLAB_UI_RESPONDED,
              "Gemini in Colab UI", "Gemini sidebar response")
if GEMINI_CLI_VERSION is not None and str(GEMINI_CLI_VERSION).strip():
    record("Gemini CLI installation", "PRESENT — USER REPORTED",
           "Cloud Shell CLI version: " + str(GEMINI_CLI_VERSION).strip()[:100])
else:
    record("Gemini CLI installation", "NOT VERIFIED",
           "Cloud Shell version check not supplied")
manual_status(GEMINI_CLI_ANSWERED,
              "Gemini CLI authentication", "Interactive Gemini CLI response")
print("Manual checks processed without blocking. Continue to Step 11 to save the report.")



## 11 — Save the final evidence report to Google Drive (with readback)

Only configuration statuses, limited benign test evidence, time and project ID are written. **Never include secrets, tokens, private project data or the contents of dataset records.**


In [ ]:
report = {
    "system": "DARA — Dual-Spine AI Research Architecture",
    "phase": "3 — AI integrations and metadata readiness",
    "run_id": RUN_ID,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "google_cloud_project": PROJECT_ID,
    "github_repo": GITHUB_REPO,
    "python_version": platform.python_version(),
    "tests": checks,
    "limitations": [
        "User-reported Gemini sidebar and Cloud Shell outcomes are not programmatically verified.",
        "BigQuery and Vertex AI checks inspect only API configuration and metadata; no datasets queried.",
        "No paid Gemini or OpenAI generation was requested.",
        "GitHub code read does not verify a connected GitHub write permission from Colab."
    ]
}
if checks["Drive readback"]["status"] != "PASS":
    raise RuntimeError("Drive readback not passed — refusing to declare report saved.")
report_path = FOLDER / ("DARA_PHASE3_VERIFICATION_" + RUN_ID + ".json")
with report_path.open("x", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)
saved = json.loads(report_path.read_text(encoding="utf-8"))
assert saved["run_id"] == RUN_ID and saved["google_cloud_project"] == PROJECT_ID
print("DARA Phase 3 report saved and read back successfully:")
print(report_path)
for k,v in checks.items():
    print("•", k, ":", v["status"])


## Follow-up

Ask ChatGPT to **inspect the newest `DARA_PHASE3_VERIFICATION_*.json` file in `DARA_ResearchOS` Google Drive** and report which services passed, were blocked or remain manual. Do **not** repeat paid-inference testing without approval.

GitHub: https://github.com/Saehon/Saeid-Homayoun/tree/main/DARA

Google Drive: https://drive.google.com/drive/folders/1OXZ6v1zB0G7wmlGezLCAvm8I4Egr2UPL
